#各月の気温・降水量・日照時間と、スーパー・百貨店での衣料品の売り上げの相関

In [1]:
#import

import pandas as pd
!conda install openpyxl

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: done

# All requested packages already installed.





==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [2]:
#データの読み込み
df_x = pd.read_csv(r'C:\Users\natsu\myworks_for_GitHub\Python-machine-learning-study-log\opendata_analysis\datafiles\crimate_data.csv',encoding="cp932")
df_y = pd.read_excel(r'C:\Users\natsu\myworks_for_GitHub\Python-machine-learning-study-log\opendata_analysis\datafiles\sales.xlsx')

display(df_x.head(7))
display(df_y.head(7))

,年月,日最高気温の平均(℃),日最高気温の平均(℃).1,日最高気温の平均(℃).2,日最低気温の平均(℃),日最低気温の平均(℃).1,日最低気温の平均(℃).2,降水量の合計(mm),降水量の合計(mm).1,降水量の合計(mm).2,...,日照時間(時間).3,平均風速(m/s),平均風速(m/s).1,平均風速(m/s).2,平均湿度(％),平均湿度(％).1,平均湿度(％).2,平均雲量(10分比),平均雲量(10分比).1,平均雲量(10分比).2
0,NaN,NaN,品質情報,均質番号,NaN,品質情報,均質番号,NaN,現象なし情報,品質情報,...,均質番号,NaN,品質情報,均質番号,NaN,品質情報,均質番号,NaN,品質情報,均質番号
1,5-Jan,10.0,8,1,2.6,8,1,77.0,0,8,...,1,3.7,8,1,47.0,8,1,4.2,8,1
2,5-Feb,9.9,8,1,2.5,8,1,48.0,0,8,...,1,3.9,8,1,45.0,8,1,6.3,8,1
3,5-Mar,13.1,8,1,5.0,8,1,71.0,0,8,...,1,3.5,8,1,49.0,8,1,6.0,8,1
4,5-Apr,19.6,8,1,10.7,8,1,81.0,0,8,...,1,3.8,8,1,54.0,8,1,5.4,8,1
5,5-May,21.9,8,1,14.1,8,1,180.5,0,8,...,1,3.9,8,1,58.0,8,1,7.2,8,1
6,5-Jun,26.7,8,1,20.2,8,1,170.5,0,8,...,1,3.0,8,1,70.0,8,1,8.9,8,1


,Unnamed: 0,2020年=100,2020年=100.1,2020年=100.2,2020年=100.3,2020年=100.4,2020年=100.5,2020年=100.6,2020年=100.7,2020年=100.8,2020年=100.9,2020年=100.10,2020年=100.11,Unnamed: 13,Unnamed: 14
0,NaN,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,NaN,NaN
1,NaN,合計,合計,合計,衣料品,衣料品,衣料品,飲食料品,飲食料品,飲食料品,その他,その他,その他,NaN,NaN
2,NaN,Total,Total,Total,Clothes,Clothes,Clothes,Food and Beverages,Food and Beverages,Food and Beverages,Others,Others,Others,NaN,NaN
3,NaN,合計,百貨店,スーパー,合計,百貨店,スーパー,合計,百貨店,スーパー,合計,百貨店,スーパー,NaN,NaN
4,年月,Total,Departmentstores,Supermarkets,Total,Departmentstores,Supermarkets,Total,Departmentstores,Supermarkets,Total,Departmentstores,Supermarkets,Month,Year
5,2005年1月,123.7,197.6,99.3,292.5,291.4,299.7,83.2,120.7,78.3,130.9,152.2,119.5,Jan,2005
6,2005年2月,97.3,148.4,80.4,181.4,178.4,190.2,75.6,120.1,69.8,105.8,137.2,89.8,Feb,2005


#目的変数データの整形

In [3]:
#目的変数データの異常値を取り除く

print(df_x.columns)

df_x.columns=['年月', '日最高気温の平均(℃)', '日最高気温の平均(℃).品質情報', '日最高気温の平均(℃).均質情報', 
    '日最低気温の平均(℃)','日最低気温の平均(℃).品質情報', '日最低気温の平均(℃).均質情報',
    '降水量の合計(mm)', '降水量の合計(mm).現象なし情報','降水量の合計(mm).品質情報', '降水量の合計(mm).均質情報',
    '日照時間(時間)', '日照時間(時間).現象なし情報', '日照時間(時間).品質情報','日照時間(時間).均質情報',
    '平均風速(m/s)', '平均風速(m/s).品質情報', '平均風速(m/s).均質情報',
    '平均湿度(％)','平均湿度(％).品質情報', '平均湿度(％).均質情報',
    '平均雲量(10分比)', '平均雲量(10分比).品質情報', '平均雲量(10分比).均質情報']


Index(['年月', '日最高気温の平均(℃)', '日最高気温の平均(℃).1', '日最高気温の平均(℃).2', '日最低気温の平均(℃)',
       '日最低気温の平均(℃).1', '日最低気温の平均(℃).2', '降水量の合計(mm)', '降水量の合計(mm).1',
       '降水量の合計(mm).2', '降水量の合計(mm).3', '日照時間(時間)', '日照時間(時間).1', '日照時間(時間).2',
       '日照時間(時間).3', '平均風速(m/s)', '平均風速(m/s).1', '平均風速(m/s).2', '平均湿度(％)',
       '平均湿度(％).1', '平均湿度(％).2', '平均雲量(10分比)', '平均雲量(10分比).1', '平均雲量(10分比).2'],
      dtype='object')


In [4]:
valid_check_cols=['日最高気温の平均(℃).品質情報','日最低気温の平均(℃).品質情報','降水量の合計(mm).品質情報','日照時間(時間).品質情報','平均風速(m/s).品質情報','平均湿度(％).品質情報','平均雲量(10分比).品質情報']

for col in valid_check_cols:
    print(df_x[col].value_counts())

日最高気温の平均(℃).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64
日最低気温の平均(℃).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64
降水量の合計(mm).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64
日照時間(時間).品質情報
8       248
5        10
品質情報      1
Name: count, dtype: int64
平均風速(m/s).品質情報
8       251
5         7
品質情報      1
Name: count, dtype: int64
平均湿度(％).品質情報
8       254
5         4
品質情報      1
Name: count, dtype: int64
平均雲量(10分比).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64


In [5]:
#目的変数は正常値、準正常値のみであるから、除外すべき行はなし
#df_x2をデータ分析用のデータフレームとする

df_x2=df_x.copy()
to_drop_x=['日最高気温の平均(℃).品質情報', '日最高気温の平均(℃).均質情報', '日最低気温の平均(℃).品質情報', '日最低気温の平均(℃).均質情報',
    '降水量の合計(mm).現象なし情報','降水量の合計(mm).品質情報', '降水量の合計(mm).均質情報',
    '日照時間(時間).現象なし情報', '日照時間(時間).品質情報','日照時間(時間).均質情報',
    '平均風速(m/s).品質情報', '平均風速(m/s).均質情報',
    '平均湿度(％).品質情報', '平均湿度(％).均質情報',
    '平均雲量(10分比).品質情報', '平均雲量(10分比).均質情報']

df_x2=df_x2.drop(columns=to_drop_x,axis=1)
df_x2=df_x2.drop(index=0,axis=0)
 
display(df_x2.head(10))
display(df_x2.tail(10))

,年月,日最高気温の平均(℃),日最低気温の平均(℃),降水量の合計(mm),日照時間(時間),平均風速(m/s),平均湿度(％),平均雲量(10分比)
1,5-Jan,10.0,2.6,77.0,200.0,3.7,47.0,4.2
2,5-Feb,9.9,2.5,48.0,148.9,3.9,45.0,6.3
3,5-Mar,13.1,5.0,71.0,175.1,3.5,49.0,6.0
4,5-Apr,19.6,10.7,81.0,216.1,3.8,54.0,5.4
5,5-May,21.9,14.1,180.5,172.3,3.9,58.0,7.2
6,5-Jun,26.7,20.2,170.5,119.3,3.0,70.0,8.9
7,5-Jul,29.1,22.6,247.5,103.9,2.9,71.0,8.9
8,5-Aug,31.8,25.1,189.5,159.9,3.4,68.0,7.6
9,5-Sep,28.2,21.8,177.5,154.2,3.6,67.0,7.1
10,5-Oct,22.3,16.6,201.5,108.3,3.3,69.0,7.3


,年月,日最高気温の平均(℃),日最低気温の平均(℃),降水量の合計(mm),日照時間(時間),平均風速(m/s),平均湿度(％),平均雲量(10分比)
249,25-Sep,30.9,23.0,203.5,158.3,2.8,77.0,7.2
250,25-Oct,22.1,15.7,106.5,74.1,2.4,78.0,8.9
251,25-Nov,17.4,8.9,17.0,160.1,2.1,64.0,4.9
252,25-Dec,12.9,4.2,36.5,171.9,2.2,59.0,4.6
253,26-Jan,11.0,1.6,7.5,224.5,2.7,44.0,3.4
254,26-Feb,13.2,3.5,58.0,156.4,2.5,58.0,5.7
255,26-Mar,15.6,7.3,122.5,169.6,2.8,60.0,6.7
256,26-Apr,21.6,12.0,162.0,162.3,3.2,68.0,7.2
257,26-May,25.8,16.0,115.5,222.2,3.1,66.0,6.5
258,26-Jun,25.5,18.2,342.5,94.4,2.7,78.0,9.0


#説明変数データの前処理

In [6]:
#説明変数の不要な行・列の削除
#df_y2をデータ分析用のデータフレームとする

df_y.columns=[
    '年月',
    '合計.全体','百貨店.全体','スーパー.全体',
    '合計.衣料品','百貨店.衣料品','スーパー.衣料品',
    '合計.飲食料品','百貨店.飲食料品','スーパー.飲食料品',
    '合計.その他','百貨店.その他','スーパー.その他',
    'Month','Year'
]


df_y2=df_y.copy()

to_drop_t=['百貨店.全体','スーパー.全体',
    '合計.衣料品','百貨店.衣料品','スーパー.衣料品',
    '合計.飲食料品','百貨店.飲食料品','スーパー.飲食料品',
    '合計.その他','百貨店.その他','スーパー.その他',
    'Month','Year'
]
df_y2=df_y2.drop(to_drop_t,axis=1)
df_y2=df_y2.drop(index=range(0,5),axis=0)

display(df_y2.head(10))
display(df_y2.tail(10))


,年月,合計.全体
5,2005年1月,123.7
6,2005年2月,97.3
7,2005年3月,111.9
8,2005年4月,110
9,2005年5月,110
10,2005年6月,109.9
11,2005年7月,123.6
12,2005年8月,104.9
13,2005年9月,101.6
14,2005年10月,112.2


,年月,合計.全体
251,2025年7月,117.1
252,2025年8月,117.2
253,2025年9月,109.4
254,2025年10月,114.4
255,2025年11月,121
256,2025年12月,144.8
257,2026年1月,120
258,2026年2月,107.3
259,2026年3月,119.3
260,2026年4月,112.2


#目的変数・説明変数双方のデータの前処理

In [7]:
print(df_x2.shape)
print(df_y2.shape)

(258, 8)
(256, 2)


In [8]:
df_x2=df_x2.drop(index=[257,258],axis=0)
print(df_x2.shape)

(256, 8)


In [9]:
#不要な行列の削除、欠損値の確認
df_y2=df_y2.drop(columns=['年月'],axis=1)
display(df_x2.isnull().sum())
display(df_y2.isnull().sum())

年月             0
日最高気温の平均(℃)    0
日最低気温の平均(℃)    0
降水量の合計(mm)     0
日照時間(時間)       0
平均風速(m/s)      0
平均湿度(％)        0
平均雲量(10分比)     0
dtype: int64

合計.全体    0
dtype: int64

In [10]:
#データ分析用の、k,tを連結したデータフレーム df3の作成
df_x2 = df_x2.reset_index(drop=True)
df_y2 = df_y2.reset_index(drop=True)

df3 = pd.concat([df_x2, df_y2], axis=1)
df3=pd.concat([df_x2,df_y2],axis=1)
df3.index=df3[['年月']]
df3=df3.drop(columns=['年月'],axis=1)
display(df3.head(10))

,日最高気温の平均(℃),日最低気温の平均(℃),降水量の合計(mm),日照時間(時間),平均風速(m/s),平均湿度(％),平均雲量(10分比),合計.全体
"(5-Jan,)",10.0,2.6,77.0,200.0,3.7,47.0,4.2,123.7
"(5-Feb,)",9.9,2.5,48.0,148.9,3.9,45.0,6.3,97.3
"(5-Mar,)",13.1,5.0,71.0,175.1,3.5,49.0,6.0,111.9
"(5-Apr,)",19.6,10.7,81.0,216.1,3.8,54.0,5.4,110
"(5-May,)",21.9,14.1,180.5,172.3,3.9,58.0,7.2,110
"(5-Jun,)",26.7,20.2,170.5,119.3,3.0,70.0,8.9,109.9
"(5-Jul,)",29.1,22.6,247.5,103.9,2.9,71.0,8.9,123.6
"(5-Aug,)",31.8,25.1,189.5,159.9,3.4,68.0,7.6,104.9
"(5-Sep,)",28.2,21.8,177.5,154.2,3.6,67.0,7.1,101.6
"(5-Oct,)",22.3,16.6,201.5,108.3,3.3,69.0,7.3,112.2


In [11]:
#重回帰分析の目的変数、説明変数の定義
x=df3.loc[:,'日最高気温の平均(℃)':'平均雲量(10分比)']
y=df3[['合計.全体']]

In [12]:
display(df3.corr())

,日最高気温の平均(℃),日最低気温の平均(℃),降水量の合計(mm),日照時間(時間),平均風速(m/s),平均湿度(％),平均雲量(10分比),合計.全体
日最高気温の平均(℃),1.000000,0.990228,0.440030,-0.168859,0.108928,0.821434,0.728640,-0.245506
日最低気温の平均(℃),0.990228,1.000000,0.461464,-0.251917,0.098064,0.810628,0.755728,-0.226888
降水量の合計(mm),0.440030,0.461464,1.000000,-0.454171,0.042315,0.545993,0.572837,-0.273835
日照時間(時間),-0.168859,-0.251917,-0.454171,1.000000,0.228220,-0.446047,-0.644350,0.129160
平均風速(m/s),0.108928,0.098064,0.042315,0.228220,1.000000,-0.188491,0.092594,-0.205316
平均湿度(％),0.821434,0.810628,0.545993,-0.446047,-0.188491,1.000000,0.783452,-0.236799
平均雲量(10分比),0.728640,0.755728,0.572837,-0.644350,0.092594,0.783452,1.000000,-0.357664
合計.全体,-0.245506,-0.226888,-0.273835,0.129160,-0.205316,-0.236799,-0.357664,1.000000


In [13]:
#テストデータの分割
train_val,test=train_test_split(df3,test_size=0.2,random_state=0)


In [14]:

'''
x_train  xの訓練データ
x_val  xの検証データ
y_train  yの訓練データ
y_val  yの検証データ
test  df3のうち2割を分割したテストデータ
'''

#訓練データと検証データへの分割
train_x=train_val.loc[:,'日最高気温の平均(℃)':'平均雲量(10分比)']
train_y=train_val[['合計.全体']]

x_train,x_val,y_train,y_val=train_test_split(train_x,train_y,test_size=0.2,random_state=0)


In [15]:
#標準化
sc_model_x,sc_model_y=StandardScaler()

sc_model_x.fit(x_train)
sc_x_train=sc_model_x.transform(x_train)
sc_x_val=sc_model_x.transform(x_val)

sc_model_y.fit(y_train)
sc_y_train=sc_model_y.transform(y_train)
sc_y_val=sc_model_y.transform(y_val)

TypeError: cannot unpack non-iterable StandardScaler object

In [ ]:
#データの学習、評価
model=LinearRegression()
model.fit(sc_x_train,sc_y_train)
model.score(sc_x_val,sc_y_val)

In [ ]:
import pickle

with open('boston.pkl','wb')as f:
    pickle.dump(model,f)

with open('boston_scx.pkl','wb')as f:
    pickle.dump(sc_model_x2,f)
    
with open('boston_scy.pkl','wb')as f:
    pickle.dump(sc_model_y2,f)

In [ ]:
#余裕があればVIF,ｐ値も調べたい